In [1]:
# 필요한 라이브러리 불러오기.
import os
import sys
import torch
import wandb

from datetime import datetime
from random import sample
from torch import nn, optim
from torch.utils.data import DataLoader, random_split, ConcatDataset
from torchvision import datasets
from torchvision.transforms import transforms, v2

BASE_PATH = '/home/ksy/Develop/Kut-Deep-Learning-250204'
sys.path.append(BASE_PATH)

from _01_code._99_common_utils.utils import get_num_cpu_cores
from _01_code._09_fcn_best_practice.c_trainer import ClassificationTrainer
from _01_code._09_fcn_best_practice.d_tester import ClassificationTester

CURRENT_FILE_PATH = f'{BASE_PATH}/_03_homeworks/homework_3'
CHECKPOINT_FILE_PATH = os.path.join(CURRENT_FILE_PATH, "checkpoints")
if not os.path.isdir(CHECKPOINT_FILE_PATH):
  os.makedirs(os.path.join(CURRENT_FILE_PATH, "checkpoints"))

In [2]:
def get_fashion_mnist_data():
    data_path = os.path.join(BASE_PATH, "_00_data", "j_fashion_mnist")

    f_mnist_train = datasets.FashionMNIST(data_path, train=True, download=True, transform=transforms.ToTensor())
    f_mnist_train, f_mnist_validation = random_split(f_mnist_train, [55_000, 5_000])

    if True:
        # 이미지 Augmentation.
        img_transform = v2.Compose([
            v2.RandomHorizontalFlip(),
            v2.RandomCrop([28, 28], padding=4),
        ])

        transformed_f_mnist_train = []
        for img, label in f_mnist_train:
            timg = img_transform(img)
            transformed_f_mnist_train.append((timg, label))
        f_mnist_train = ConcatDataset([f_mnist_train, transformed_f_mnist_train])

    # mean std
    if True:
        # 이미지 Augmentation을 진행했으므로 자동으로 mean, std를 계산할 수 있도록 함.
        train_imgs = torch.stack([i for i, _ in f_mnist_train], dim=3)
        train_stat = (train_imgs.view(1, -1).mean(), train_imgs.view(1, -1).std())
        print(f'f_mnist_train:')
        print(f'    mean: {train_stat[0]}')
        print(f'    std:  {train_stat[1]}')
    else:
        # 원본 데이터의 mean과 std.
        # f_mnist_train:
        #     mean: 0.28632092475891113
        #     std:  0.353121280670166
        train_stat = (0.28632092475891113, 0.353121280670166)

    print("Num Train Samples: ", len(f_mnist_train))
    print("Num Validation Samples: ", len(f_mnist_validation))
    print(f"Stat Train Samples: (mean, std): {train_stat}")
    print("Sample Data Shape: ", f_mnist_train[0][0].shape)  # torch.Size([1, 28, 28])
    print("Sample Data Target: ", f_mnist_train[0][1])  # 9

    num_data_loading_workers = get_num_cpu_cores()
    print("Number of Data Loading Workers:", num_data_loading_workers)

    train_data_loader = DataLoader(
        dataset=f_mnist_train, batch_size=wandb.config.batch_size, shuffle=True,
        pin_memory=True, num_workers=num_data_loading_workers
    )

    validation_data_loader = DataLoader(
        dataset=f_mnist_validation, batch_size=wandb.config.batch_size,
        pin_memory=True, num_workers=num_data_loading_workers
    )

    f_mnist_transforms = nn.Sequential(
        transforms.ConvertImageDtype(torch.float),
        transforms.Normalize(mean=train_stat[0], std=train_stat[1]),
    )

    return train_data_loader, validation_data_loader, f_mnist_transforms

In [3]:
def get_fashion_mnist_test_data():
    data_path = os.path.join(BASE_PATH, "_00_data", "j_fashion_mnist")

    f_mnist_test_images = datasets.FashionMNIST(data_path, train=False, download=True)
    f_mnist_test = datasets.FashionMNIST(data_path, train=False, download=True, transform=transforms.ToTensor())

    # mean std
    if False:
        test_imgs = torch.stack([i for i, _ in f_mnist_test], dim=3)
        print(f'f_mnist_test:')
        print(f'    mean: {test_imgs.view(1, -1).mean()}')
        print(f'    std:  {test_imgs.view(1, -1).std()}')

        print('[INFO] Calculating mean, std done. quitting...')
        exit(0)
    else:
        # f_mnist_train:
        #     mean: 0.2868492901325226
        #     std:  0.3524441719055176
        test_stat = (0.2868492901325226, 0.3524441719055176)

    print("Num Test Samples: ", len(f_mnist_test))
    print(f"Stat Test Samples: (mean, std): {test_stat}")
    print("Sample Shape: ", f_mnist_test[0][0].shape)  # torch.Size([1, 28, 28])

    test_data_loader = DataLoader(dataset=f_mnist_test, batch_size=len(f_mnist_test))

    f_mnist_transforms = nn.Sequential(
        transforms.ConvertImageDtype(torch.float),
        transforms.Normalize(mean=test_stat[0], std=test_stat[1]),
    )

    return f_mnist_test_images, test_data_loader, f_mnist_transforms

In [4]:
def get_cnn_model():
  class MyModel(nn.Module):
    def __init__(self, in_channels, n_output):
      super().__init__()

      self.model = nn.Sequential(
        # B x 1 x 28 x 28 --> B x 6 x (28 - 5 + 1) x (28 - 5 + 1) = B x 6 x 24 x 24
        nn.Conv2d(in_channels=in_channels, out_channels=6, kernel_size=(5, 5), stride=(1, 1)),
        # B x 6 x 24 x 24 --> B x 6 x 12 x 12
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.ReLU(),
        # B x 6 x 12 x 12 --> B x 16 x (12 - 5 + 1) x (12 - 5 + 1) = B x 16 x 8 x 8
        nn.Conv2d(in_channels=6, out_channels=16, kernel_size=(5, 5), stride=(1, 1)),
        # B x 16 x 8 x 8 --> B x 16 x 4 x 4
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.ReLU(),
        # B x 16 x 4 x 4 --> B x 256
        nn.Flatten(),
        nn.Linear(256, 128),
        nn.ReLU(),
        nn.Linear(128, n_output),
      )

    def forward(self, x):
      x = self.model(x)
      return x

  # 1 * 28 * 28
  my_model = MyModel(in_channels=1, n_output=10)

  return my_model

In [5]:
def train(args):
  run_time_str = datetime.now().astimezone().strftime('%Y-%m-%d_%H-%M-%S')

  config = {
    'epochs': args.epochs,
    'batch_size': args.batch_size,
    'validation_intervals': args.validation_intervals,
    'learning_rate': args.learning_rate,
    'early_stop_patience': args.early_stop_patience,
    'early_stop_delta': args.early_stop_delta
  }

  project_name = "cnn_fashion_mnist"
  project_name = args.project_name

  device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
  print(f"Training on device {device}.")
  
  wandb.init(
    mode="online" if args.wandb else "disabled",
    project=project_name,
    notes="mnist experiment with cnn",
    tags=["cnn", "mnist"],
    name=run_time_str,
    config=config
  )
  print(args)
  print(wandb.config)

  train_data_loader, validation_data_loader, f_mnist_transforms = get_fashion_mnist_data()
  model = get_cnn_model()
  model.to(device)

  from torchinfo import summary
  summary(model=model, device=device, input_size=(1, 1, 28, 28))

  optimizer = optim.SGD(model.parameters(), lr=wandb.config.learning_rate)

  classification_trainer = ClassificationTrainer(
    project_name, model, optimizer, train_data_loader, validation_data_loader, f_mnist_transforms,
    run_time_str, wandb, device, CHECKPOINT_FILE_PATH
  )
  classification_trainer.train_loop()

  wandb.finish()

In [6]:
def test(args):
  model = get_cnn_model()

  test_imgs, test_data_loader, f_mnist_test_transforms = get_fashion_mnist_test_data()
  classification_tester = ClassificationTester(
    args.project_name, model, test_data_loader, f_mnist_test_transforms, CHECKPOINT_FILE_PATH
  )
  
  for imgs, targets in test_data_loader:
    combined = list(zip([img for img, _ in test_imgs], imgs, targets))
    
    while True:
      xs = []
      ys = []
      zs = []

      cases = sample(combined, 5)
      for img, img_tensor, target in cases:
        img_tensor = img_tensor.unsqueeze(dim=1)
        pred = classification_tester.test_single(img_tensor)
        
        xs.append(pred)
        ys.append(int(target))
        zs.append(img)
        
      if any(x != y for x, y in zip(xs, ys)):
          print(xs)
          print(ys)
          print(zs)
          break

In [ ]:
# dict에 접근할 때 `.` 연산자로 접근할 수 있도록 하는 dict.
# stolen from: https://stackoverflow.com/questions/2352181/how-to-use-a-dot-to-access-members-of-dictionary
class dotdict(dict):
    """dot.notation access to dictionary attributes"""
    __getattr__ = dict.get
    __setattr__ = dict.__setitem__
    __delattr__ = dict.__delitem__

args = dotdict({
    'wandb': True,
    'epochs': 10000,
    'batch_size': 512,
    'validation_intervals': 10,
    'learning_rate': 0.001,
    'early_stop_patience': 10,
    'early_stop_delta': 1e-5,
    'project_name': 'cnn_fashion_mnist',
})

# train(args)
test(args)
print('done')

Num Test Samples:  10000
Stat Test Samples: (mean, std): (0.2868492901325226, 0.3524441719055176)
Sample Shape:  torch.Size([1, 28, 28])
MODEL FILE: /home/ksy/Develop/Kut-Deep-Learning-250204/_03_homeworks/homework_3/checkpoints/cnn_fashion_mnist_checkpoint_latest.pt
[3, 6, 3, 4, 1]
[1, 0, 3, 4, 1]
[<PIL.Image.Image image mode=L size=28x28 at 0x75666C7ED700>, <PIL.Image.Image image mode=L size=28x28 at 0x75666C61DE80>, <PIL.Image.Image image mode=L size=28x28 at 0x75666C9FE9C0>, <PIL.Image.Image image mode=L size=28x28 at 0x75666C6CE7E0>, <PIL.Image.Image image mode=L size=28x28 at 0x75666C8AC6B0>]
